# Step 3: Evaluate Under Distribution Shift

**CMPE 260 — Group 6**

This notebook evaluates trained agents (from Steps 1 & 2) under gravity shift and observation noise.

**Prerequisites:** Run `01_train_baseline.ipynb` and `02_train_ensemble.ipynb` first, or upload checkpoints.

**Runtime:** ~10 min total (evaluation is fast — no training)

**Outputs:**
- `results/shift_{env}_{2Q|3Q}.csv` for each environment and critic config

In [ ]:
!git clone -b sp1ffygeek_check_3 https://github.com/shloakk/iql-robustness-analysis.git
%cd iql-robustness-analysis

!pip install -q jax jaxlib flax optax
!pip install -q mujoco "gymnasium[mujoco]" gym
!pip install -q h5py tqdm matplotlib numpy scipy
!pip install -q absl-py ml_collections tensorboardX tensorflow-probability
!pip install -q git+https://github.com/Farama-Foundation/d4rl@master

In [ ]:
import os, sys, csv
import numpy as np

sys.path.insert(0, '.')
import wrappers, gym, d4rl
from wrappers.gravity_shift import GravityShift
from wrappers.observation_noise import ObservationNoise
from iql import Learner
from iql.dataset_utils import D4RLDataset, split_into_trajectories
from evaluation import evaluate

# ============================================================
# CONFIGURATION
# ============================================================
ENVIRONMENTS = ['hopper-medium-v2', 'halfcheetah-medium-v2', 'walker2d-medium-v2']
CRITIC_CONFIGS = [2, 3]  # DoubleCritic and TripleCritic
GRAVITY_LEVELS = [0.5, 1.0, 1.5, 2.0]
OBS_NOISE_LEVELS = [0.0, 0.01, 0.1, 0.3]
MAX_STEPS = 300_000
EVAL_EPISODES = 10
SEED = 42
CONFIG = {
    'actor_lr': 3e-4, 'value_lr': 3e-4, 'critic_lr': 3e-4,
    'hidden_dims': (256, 256), 'discount': 0.99,
    'expectile': 0.7, 'temperature': 3.0,
    'dropout_rate': None, 'tau': 0.005,
}

def normalize(dataset):
    trajs = split_into_trajectories(
        dataset.observations, dataset.actions, dataset.rewards,
        dataset.masks, dataset.dones_float, dataset.next_observations)
    def compute_returns(traj):
        return sum(rew for _, _, rew, _, _, _ in traj)
    trajs.sort(key=compute_returns)
    dataset.rewards /= compute_returns(trajs[-1]) - compute_returns(trajs[0])
    dataset.rewards *= 1000.0

In [ ]:
# Option A: Train fresh (if no checkpoints uploaded)
# Option B: Upload checkpoints from Steps 1 & 2
#
# For Option A, we train inline. For Option B, upload checkpoint
# folders to tmp/{env_name}_{N}Q/checkpoint_{MAX_STEPS}/

TRAIN_FRESH = True  # Set to False if uploading checkpoints

def get_trained_agent(env_name, num_critics, env):
    """Get a trained agent — either train fresh or load checkpoint."""
    agent = Learner(
        SEED, env.observation_space.sample()[np.newaxis],
        env.action_space.sample()[np.newaxis],
        max_steps=MAX_STEPS, num_critics=num_critics, **CONFIG)

    ckpt_dir = f'tmp/{env_name}_{num_critics}Q/checkpoint_{MAX_STEPS}'
    if os.path.exists(ckpt_dir) and not TRAIN_FRESH:
        print(f"  Loading checkpoint from {ckpt_dir}")
        agent.actor = agent.actor.load(os.path.join(ckpt_dir, 'actor.pkl'))
        agent.critic = agent.critic.load(os.path.join(ckpt_dir, 'critic.pkl'))
        agent.value = agent.value.load(os.path.join(ckpt_dir, 'value.pkl'))
        agent.target_critic = agent.target_critic.load(
            os.path.join(ckpt_dir, 'target_critic.pkl'))
    else:
        print(f"  Training fresh ({num_critics}Q, {MAX_STEPS} steps)...")
        dataset = D4RLDataset(env)
        normalize(dataset)
        from tqdm import tqdm
        for i in tqdm(range(1, MAX_STEPS + 1), desc=f'{env_name} {num_critics}Q'):
            batch = dataset.sample(256)
            agent.update(batch)
        save_dir = f'tmp/{env_name}_{num_critics}Q'
        agent.save(save_dir, MAX_STEPS)

    return agent

In [ ]:
def evaluate_under_shift(agent, env_name, shift_type, shift_levels):
    """Evaluate agent under distribution shift."""
    results = []
    for level in shift_levels:
        shifted_env = gym.make(env_name)
        if shift_type == 'gravity':
            shifted_env = GravityShift(shifted_env, gravity_scale=level)
        elif shift_type == 'obs_noise':
            shifted_env = ObservationNoise(shifted_env, noise_std=level)
        shifted_env = wrappers.EpisodeMonitor(shifted_env)
        shifted_env = wrappers.SinglePrecision(shifted_env)
        shifted_env.seed(SEED)

        stats = evaluate(agent, shifted_env, EVAL_EPISODES)
        results.append({
            'shift_type': shift_type, 'shift_level': level,
            'raw_return': stats['return'], 'episode_length': stats['length'],
        })
        print(f"    {shift_type}={level:.2f}: return={stats['return']:.2f}")
    return results

def compute_robustness_drop(baseline, shifted):
    if baseline == 0:
        return float('inf') if shifted != 0 else 0.0
    return (baseline - shifted) / abs(baseline)

In [ ]:
os.makedirs('results', exist_ok=True)

for env_name in ENVIRONMENTS:
    for num_critics in CRITIC_CONFIGS:
        label = f"{num_critics}Q"
        print(f"\n{'='*60}")
        print(f"Evaluating {label} on {env_name}")
        print(f"{'='*60}")

        env = gym.make(env_name)
        env = wrappers.EpisodeMonitor(env)
        env = wrappers.SinglePrecision(env)
        env.seed(SEED)

        agent = get_trained_agent(env_name, num_critics, env)

        # Evaluate under gravity shift
        print(f"  Gravity shift:")
        gravity_results = evaluate_under_shift(agent, env_name, 'gravity', GRAVITY_LEVELS)

        # Evaluate under observation noise
        print(f"  Observation noise:")
        noise_results = evaluate_under_shift(agent, env_name, 'obs_noise', OBS_NOISE_LEVELS)

        # Add robustness drop
        all_results = gravity_results + noise_results
        for r in all_results:
            if r['shift_type'] == 'gravity':
                baseline = next(x['raw_return'] for x in gravity_results if x['shift_level'] == 1.0)
            else:
                baseline = next(x['raw_return'] for x in noise_results if x['shift_level'] == 0.0)
            r['robustness_drop'] = compute_robustness_drop(baseline, r['raw_return'])

        # Save
        outfile = f'results/shift_{env_name}_{label}.csv'
        with open(outfile, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=[
                'shift_type', 'shift_level', 'raw_return',
                'episode_length', 'robustness_drop'])
            w.writeheader()
            w.writerows(all_results)
        print(f"  Saved: {outfile}")

print("\n✅ All shift evaluations complete!")

In [ ]:
from google.colab import files
import glob
for f in glob.glob('results/shift_*.csv'):
    files.download(f)